In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sb
import skimage
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scipy.ndimage as ndimage

import mnds

In [ ]:
# Change dir accordingly:
images_dir = '/home/caicedo/scr/jcaicedo/Micronuclei-data/dataset_v2/'

files = os.listdir(images_dir)
annot_files = [x for x in files if x.endswith('png')]

In [ ]:
def crop_objects(im, labels, mni, margin=16):
    m = margin
    crops = []
    for k,r in mni.iterrows():
        ys, ye = max(r.y - m, 0) , min(r.y + m, im.shape[0])
        xs, xe = max(r.x - m, 0) , min(r.x + m, im.shape[1])
        c = im[ys:ye, xs:xe].copy()
        l = labels[ys:ye, xs:xe] == labels[r.y, r.x]
        c = c*l
        crops.append(c)
    return crops

In [ ]:
count_annotations = 0
mni_dfs = []
crops = []
for fname in annot_files:
    imid = fname.split('.')[0]
    print(imid)
    im = mnds.read_image(images_dir, imid, 'phenotype.tif')
    mni, labels = mnds.read_micronuclei_annotations(images_dir, imid, scale_factor=1.0)
    
    count_annotations += len(mni)
    
    print(f"{imid}: micronuclei:{len(mni)}")
    
    mni["Image"] = imid
    mni_dfs.append(mni)
    
    crops += crop_objects(im, labels, mni)
    
print("Total micronuclei:",count_annotations)

In [ ]:
MNI = pd.concat(mni_dfs).reset_index(drop=True)
MNI = MNI.sort_values(by="area")

In [ ]:
bins = [x for x in range(0,400,4)]
sb.histplot(data=MNI, x="area", bins=bins)

In [ ]:
bins = [x for x in range(0,800,10)]
sb.histplot(data=MNI, x="area", bins=bins)

In [ ]:
for k in MNI[MNI.area > 400].index:
    print(k, MNI.loc[k,"area"])
    plt.imshow(crops[k])
    plt.show()
    break

In [ ]:
def mean_object(a,b, show=True):
    mean = np.zeros_like(crops[0])
    count = 0
    for k in MNI[(MNI.area >= a) & (MNI.area < b)].index:
        if crops[k].shape == (32,32):
            mean += crops[k]
            count += 1
    mean = mean / count
    if show:
        plt.imshow(mean)
        plt.show()
    return mean

In [ ]:
templates = []
k = 0
for j in range(10,100,10):
    p = np.percentile(MNI.area, j)
    print(k,p)
    t = mean_object(k,p)
    templates.append(t)
    k = p

In [ ]:
for k in MNI.index:
    c = crops[k]
    if c.shape[0] < 64 or c.shape[1] < 64:
        print(k, MNI.iloc[k], c.shape)
        plt.imshow(c)
        plt.show()

In [ ]:
result = skimage.feature.match_template(im, templates[0])
result = np.zeros_like(r)
for t in templates:
    r = skimage.feature.match_template(im, t)
    r[r < np.percentile(r, 99.9)] = 0
    result += r

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(result)

In [ ]:
r.shape, im.shape

In [ ]:
b = plt.hist(r.flatten(), bins=100)
np.percentile(r, 99)